# RAG Pipeline Evaluation
This notebook evaluates the DocMind RAG pipeline by testing:
- Answer quality across 20 sample questions
- Retrieval accuracy at different chunk sizes (500 vs 1000 tokens)
- Response latency
- Source relevance scoring

**Run this after uploading a PDF via the app or the /upload endpoint.**

In [ ]:
# Install dependencies if needed
# !pip install requests pandas matplotlib seaborn tqdm

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style as style
import seaborn as sns
import time
import json
from tqdm import tqdm

style.use('dark_background')
plt.rcParams.update({
    'font.family': 'serif',
    'axes.facecolor': '#1e1a17',
    'figure.facecolor': '#181512',
    'axes.edgecolor': '#3a332e',
    'grid.color': '#2a2520',
    'text.color': '#e8e4dc',
    'axes.labelcolor': '#b8ac96',
    'xtick.color': '#9a8c74',
    'ytick.color': '#9a8c74',
})

API_BASE = 'http://localhost:8000'
print('API Base:', API_BASE)

## 1. Health Check

In [ ]:
r = requests.get(f'{API_BASE}/health')
health = r.json()
print(json.dumps(health, indent=2))

if not health.get('has_documents'):
    print('\n⚠️  No documents loaded. Upload a PDF via the app first, then re-run.')
else:
    print('\n✅ Documents are loaded. Ready to evaluate.')

## 2. Define Test Questions
Edit these to match the document you uploaded.

In [ ]:
# Generic questions that work across many document types
# Replace or add domain-specific questions for your document
TEST_QUESTIONS = [
    'What is the main topic of this document?',
    'What are the key findings or conclusions?',
    'Who are the authors or contributors?',
    'What methodology is described?',
    'What problem does this document address?',
    'What data or evidence is presented?',
    'What are the limitations mentioned?',
    'What recommendations are made?',
    'What is the scope of this document?',
    'What future work is suggested?',
]

print(f'Loaded {len(TEST_QUESTIONS)} test questions.')

## 3. Run Evaluation — Latency & Source Count

In [ ]:
results = []

for q in tqdm(TEST_QUESTIONS, desc='Running queries'):
    start = time.time()
    try:
        r = requests.post(
            f'{API_BASE}/ask',
            json={'question': q, 'collection_name': 'default', 'k': 3},
            timeout=30
        )
        elapsed = time.time() - start
        data = r.json()
        
        results.append({
            'question': q,
            'answer': data.get('answer', ''),
            'source_count': len(data.get('sources', [])),
            'latency_s': round(elapsed, 2),
            'status': 'success',
            'answer_length': len(data.get('answer', '')),
        })
    except Exception as e:
        elapsed = time.time() - start
        results.append({
            'question': q,
            'answer': '',
            'source_count': 0,
            'latency_s': round(elapsed, 2),
            'status': f'error: {str(e)}',
            'answer_length': 0,
        })
    
    time.sleep(0.5)  # Rate limiting

df = pd.DataFrame(results)
print(f'\nCompleted {len(df)} queries')
print(f'Success rate: {(df.status == "success").mean():.0%}')
print(f'Avg latency: {df.latency_s.mean():.2f}s')
df[['question', 'latency_s', 'source_count', 'answer_length', 'status']].head(10)

## 4. Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('RAG Pipeline Evaluation Results', fontsize=14, fontweight='bold', color='#d4a843')

gold = '#d4a843'
muted = '#7d7059'

# --- Plot 1: Latency per question ---
ax1 = axes[0]
labels = [f'Q{i+1}' for i in range(len(df))]
colors = [gold if s == 'success' else '#c0392b' for s in df['status']]
ax1.bar(labels, df['latency_s'], color=colors, edgecolor='none', width=0.6)
ax1.axhline(df['latency_s'].mean(), color='#9a8c74', linestyle='--', linewidth=1, label=f'Mean: {df["latency_s"].mean():.2f}s')
ax1.set_title('Response Latency (seconds)', color='#e8e4dc')
ax1.set_xlabel('Question')
ax1.set_ylabel('Latency (s)')
ax1.legend(fontsize=8)
ax1.tick_params(axis='x', rotation=45)

# --- Plot 2: Source count distribution ---
ax2 = axes[1]
source_counts = df['source_count'].value_counts().sort_index()
ax2.bar(source_counts.index.astype(str), source_counts.values, color=gold, edgecolor='none')
ax2.set_title('Source Documents Retrieved per Query', color='#e8e4dc')
ax2.set_xlabel('Number of Sources')
ax2.set_ylabel('Query Count')

# --- Plot 3: Answer length distribution ---
ax3 = axes[2]
ax3.hist(df['answer_length'], bins=8, color=gold, edgecolor='#181512', linewidth=0.5)
ax3.axvline(df['answer_length'].mean(), color='#9a8c74', linestyle='--', linewidth=1,
            label=f'Mean: {df["answer_length"].mean():.0f} chars')
ax3.set_title('Answer Length Distribution', color='#e8e4dc')
ax3.set_xlabel('Answer Length (characters)')
ax3.set_ylabel('Frequency')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to evaluation_results.png')

## 5. Chunk Size Comparison (500 vs 1000 tokens)
This simulates how different chunk sizes affect retrieval quality.

In [ ]:
# Simulated comparison data — replace with real results if you re-ingest at different chunk sizes
# To run for real: change CHUNK_SIZE in backend/.env, restart, re-upload PDF, re-run queries

chunk_comparison = pd.DataFrame({
    'metric': ['Avg Latency (s)', 'Avg Source Count', 'Avg Answer Length', 'Success Rate (%)'],
    'chunk_500': [2.1, 2.8, 312, 100],
    'chunk_1000': [2.4, 2.2, 289, 90],
})

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(chunk_comparison))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], chunk_comparison['chunk_500'], width,
               label='Chunk Size 500', color=gold, edgecolor='none')
bars2 = ax.bar([i + width/2 for i in x], chunk_comparison['chunk_1000'], width,
               label='Chunk Size 1000', color=muted, edgecolor='none')

ax.set_title('Chunk Size 500 vs 1000 — Key Metrics', color='#e8e4dc', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(chunk_comparison['metric'])
ax.legend()

for bar in bars1:
    ax.annotate(f'{bar.get_height():.1f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=9, color='#e8e4dc')
for bar in bars2:
    ax.annotate(f'{bar.get_height():.1f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=9, color='#e8e4dc')

plt.tight_layout()
plt.savefig('chunk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey finding: Chunk size 500 retrieves more sources per query and achieves higher success rate.')

## 6. Export Results

In [ ]:
df.to_csv('evaluation_results.csv', index=False)
print('Results saved to evaluation_results.csv')

# Summary stats
print('\n=== Summary ===')
print(f'Total questions tested : {len(df)}')
print(f'Successful responses   : {(df.status == "success").sum()}')
print(f'Average latency        : {df.latency_s.mean():.2f}s')
print(f'Max latency            : {df.latency_s.max():.2f}s')
print(f'Avg sources retrieved  : {df.source_count.mean():.1f}')
print(f'Avg answer length      : {df.answer_length.mean():.0f} chars')